# 08 — Çıkarım (Inference) Pipeline'ı

Bu notebook, **ham bir DICOM arşivinden sonuç görüntülerine** giden uçtan uca akışı
gösterir. Masaüstü arayüzü (`app/lungscan_gui.py`) bu akışın üretim sürümüdür;
buradaki kod Colab ortamında çalışan prototiptir.

## Akış

```
DICOM arşivi
   │  1. DICOM → PNG (InstanceNumber sırasına göre)
   ▼
PNG kesitler
   │  2. Sınıflandırıcı (model_in_5_1.h5) ile normal / anormal ayrımı
   ▼
normal/  +  abnormal/
              │  3. Segmentasyon (best.pt) ile lezyon konturları
              ▼
           segmented/
              │  4. normal/ ve segmented/ tek klasörde birleştirilir
              ▼
           output/
              │  5. PNG → DICOM (orijinal seri ile aynı InstanceNumber sırası)
              ▼
        hasta_sonuclari/
              │  6. Dışa aktarılan DICOM serisinin doğrulanması
              ▼
```

## Tasarım kararları

* **Yıkıcı olmayan akış:** Önceki sürümde ara adımlar kaynak klasördeki DICOM
  dosyalarını silerek ilerliyordu (`os.remove`, `shutil.rmtree`). Tek bir hata
  durumunda orijinal veri kaybolurdu. Bu sürümde kaynak klasöre **hiç yazılmaz**;
  tüm çıktılar `lungscan/` altında toplanır.
* **Sıralama:** Kesit sırası `InstanceNumber` etiketiyle belirlenir. Etiket
  okunamazsa dosya sırası korunur ve durum raporlanır.
* **Sınıf eşlemesi:** `ImageDataGenerator` sınıf adlarını alfabetik sıralar:
  `abnormal → 0`, `normal → 1`. Bu eşleme model dosyasıyla birlikte saklanmalıdır;
  ters çevrilirse tüm sonuçlar anlamsız olur.


## 1. Ortam


In [ ]:
# Colab kullanıyorsanız:
# from google.colab import drive
# drive.mount("/content/drive")
# %pip install -q pydicom ultralytics tensorflow

import os
import shutil

import copy

import cv2
import numpy as np
import pydicom
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.models import load_model
from ultralytics import YOLO

# ---------------------------------------------------------------- yollar
DATA_ROOT = "/content/drive/MyDrive/lungscan"
DICOM_ARCHIVE = "/content/drive/MyDrive/kaynak_klasor"       # ham DICOM arşivi
CLASSIFIER_PATH = os.path.join(DATA_ROOT, "models/model_in_5_1.h5")
SEGMENTER_PATH = os.path.join(DATA_ROOT, "models/best.pt")

WORK_DIR = os.path.join(DATA_ROOT, "cikarim")
PNG_DIR = os.path.join(WORK_DIR, "png")
NORMAL_DIR = os.path.join(WORK_DIR, "normal")
ABNORMAL_DIR = os.path.join(WORK_DIR, "abnormal")
SEGMENTED_DIR = os.path.join(WORK_DIR, "segmented")
OUTPUT_DIR = os.path.join(WORK_DIR, "output")
RESULT_DICOM_DIR = os.path.join(WORK_DIR, "hasta_sonuclari")

# ------------------------------------------------------------- parametreler
CLASSIFIER_SIZE = (256, 256)
CONFIDENCE_THRESHOLD = 0.5      # sigmoid çıkışı için karar eşiği
YOLO_CONF = 0.25
CLASS_NAMES = ("abnormal", "normal")     # ImageDataGenerator sırası (alfabetik)

for _d in (PNG_DIR, NORMAL_DIR, ABNORMAL_DIR, SEGMENTED_DIR, OUTPUT_DIR, RESULT_DICOM_DIR):
    os.makedirs(_d, exist_ok=True)


## 2. DICOM → PNG

Kesitler `InstanceNumber` etiketine göre sıralanır. Arşiv alt klasörlere dağılmış
olsa bile tüm dosyalar özyinelemeli olarak bulunur; klasör taşıma/temizleme gibi
yıkıcı adımlara gerek kalmaz.

> **Önceki sürümdeki hata:** Kaynak klasör yalnızca `DICOM` adlı bir alt klasör
> içerdiği varsayımıyla "diğer her şeyi sil" mantığıyla temizleniyor, ardından
> belirtilen aralık dışındaki DICOM dosyaları `os.remove` ile siliniyordu. Aynı
> akış, aralık dışındaki kesitleri **kaybediyordu** (bu kesitler sınıflandırıcı
> tarafından taranamaz). Bu sürüm aralık **seçer**, dosya silmez.


In [ ]:
def dicom_to_png(dicom_root, out_dir, slice_range=None):
    """DICOM serisini InstanceNumber sırasına göre PNG'ye çevirir.

    Args:
        dicom_root:  DICOM dosyalarını barındıran klasör (alt klasörler taranır).
        out_dir:     PNG'lerin yazılacağı klasör.
        slice_range: (ilk, son) 1 tabanlı indeks aralığı; None ise tüm kesitler.

    Returns:
        Yazılan PNG dosyalarının yolları (kesit sırasına göre).
    """
    paths = []
    for root, _dirs, files in os.walk(dicom_root):
        for name in files:
            if not name.startswith("."):
                paths.append(os.path.join(root, name))
    if not paths:
        raise FileNotFoundError(f"DICOM dosyası bulunamadı: {dicom_root}")

    def instance_number(path):
        try:
            dataset = pydicom.dcmread(path, stop_before_pixels=True)
            return int(dataset.get("InstanceNumber", 0) or 0)
        except Exception:
            return 10 ** 9          # okunamayan dosya en sona

    paths.sort(key=instance_number)

    if slice_range is not None:
        first, last = slice_range
        paths = paths[first - 1:last]

    written = []
    for path in paths:
        dataset = pydicom.dcmread(path)
        frame = Image.fromarray(dataset.pixel_array)
        out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(path))[0] + ".png")
        frame.save(out_path, "PNG")
        written.append(out_path)

    print(f"{len(written)} kesit PNG'ye çevrildi -> {out_dir}")
    return written


png_files = dicom_to_png(DICOM_ARCHIVE, PNG_DIR)
print("İlk kesitler:", [os.path.basename(p) for p in png_files[:3]])


## 3. Sınıflandırma: normal / anormal ayrımı

Her kesit `model_in_5_1.h5` ile değerlendirilir ve tahmin edilen sınıfın klasörüne
kopyalanır. Kopyalama (`copy`) kullanılır, taşıma (`move`) değil; böylece ara
adımlar tekrarlanabilir kalır.

> **Önceki sürümdeki hata:** `np.argmax(prediction) == 0` koşulunun yanındaki yorum
> *"Normal sınıfı"* yazıyordu, ancak `0` indeksi alfabetik sıralamada
> **`abnormal`** sınıfına karşılık gelir. Kod doğru, yorum yanlıştı; bu tür bir
> karışıklık sınıf eşlemesinin ters çevrilmesine ve tüm sonuçların bozulmasına yol
> açabilir. Bu sürümde eşleme tek bir sabitten (`CLASS_NAMES`) okunur.


In [ ]:
def classify_slices(model_path, source_dir, normal_dir, abnormal_dir,
                    class_names=CLASS_NAMES, input_size=CLASSIFIER_SIZE,
                    threshold=CONFIDENCE_THRESHOLD):
    """Kesitleri sınıflandırıcıdan geçirir ve sınıf klasörlerine kopyalar.

    Returns:
        {"normal": n, "abnormal": m} sayıları.
    """
    model = load_model(model_path)
    counts = {name: 0 for name in class_names}

    for filename in sorted(f for f in os.listdir(source_dir) if f.endswith(".png")):
        image = Image.open(os.path.join(source_dir, filename)).convert("RGB")
        image = image.resize(input_size)

        array = np.expand_dims(np.array(image) / 255.0, axis=0)
        probabilities = model.predict(array, verbose=0)[0]

        # Giriş boyutuna göre çıkış biçimi değişebilir:
        #   (2,) → softmax,  (1,) → sigmoid
        if probabilities.shape[0] == 2:
            predicted = int(np.argmax(probabilities))
        else:
            predicted = int(probabilities[0] >= threshold)

        name = class_names[predicted]
        target_dir = abnormal_dir if name == "abnormal" else normal_dir
        shutil.copy(os.path.join(source_dir, filename), os.path.join(target_dir, filename))
        counts[name] += 1

    print(f"Sınıflandırma tamamlandı: {counts}")
    return counts


counts = classify_slices(CLASSIFIER_PATH, PNG_DIR, NORMAL_DIR, ABNORMAL_DIR)


## 4. Segmentasyon: lezyon konturları

Yalnızca **anormal** olarak sınıflandırılan kesitler segmentasyon modelinden geçirilir.
Konturlar 1 piksel kalınlığında yeşil çizilir; lezyonun içi boyanmaz.


In [ ]:
def segment_abnormal_slices(model_path, source_dir, output_dir,
                            conf=YOLO_CONF, colour=(0, 255, 0)):
    """Anormal kesitlerde lezyon konturlarını çizip `output_dir`'e yazar."""
    model = YOLO(model_path)
    written = 0

    for filename in sorted(f for f in os.listdir(source_dir) if f.endswith(".png")):
        image = cv2.imread(os.path.join(source_dir, filename))
        if image is None:
            continue
        height, width = image.shape[:2]

        result = model.predict(image, conf=conf, verbose=False)[0]
        canvas = image.copy()

        if result.masks is not None:
            for mask in result.masks.data.cpu().numpy():
                mask = cv2.resize(mask, (width, height))
                binary = (mask > 0.5).astype(np.uint8) * 255
                contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(canvas, contours, -1, colour, 1)

        cv2.imwrite(os.path.join(output_dir, filename), canvas)
        written += 1

    print(f"{written} kesit segmentlendi -> {output_dir}")
    return written


segmented_count = segment_abnormal_slices(SEGMENTER_PATH, ABNORMAL_DIR, SEGMENTED_DIR)


## 5. Normal ve işaretli kesitleri birleştirme

Sonuç serisi, **tüm** kesitleri içermelidir: normal kesitler işaretsiz, anormal
kesitler konturlu. Aksi hâlde oluşturulan DICOM serisi orijinal kesit sayısıyla
eşleşmez ve görüntüleme yazılımında seri bozuk görünür.


In [ ]:
def merge_outputs(normal_dir, segmented_dir, output_dir):
    """Normal ve segmentlenmiş kesitleri tek klasörde birleştirir."""
    os.makedirs(output_dir, exist_ok=True)
    copied = 0

    for source in (normal_dir, segmented_dir):
        for filename in sorted(f for f in os.listdir(source) if f.endswith(".png")):
            shutil.copy(os.path.join(source, filename), os.path.join(output_dir, filename))
            copied += 1

    print(f"{copied} kesit birleştirildi -> {output_dir}")
    return copied


merged_count = merge_outputs(NORMAL_DIR, SEGMENTED_DIR, OUTPUT_DIR)


## 6. PNG → DICOM

Sonuçlar, orijinal DICOM serisinin **meta verileri korunarak** DICOM'a çevrilir:
hasta bilgileri, çekim parametreleri ve `InstanceNumber` sırası korunur. Böylece
çıktı, hastane görüntüleme yazılımında (PACS) orijinal seri gibi açılabilir.

> **Önceki sürümdeki hata:** `new_dicom.Rows, new_dicom.Columns = png_image.size`
> satırı yanlıştı. Pillow'un `.size` özelliği `(genişlik, yükseklik)` sırasındadır,
> DICOM'da ise `Rows` **yükseklik**, `Columns` **genişlik**tir. Kare görüntülerde
> (512×512) sonuç doğru olduğu için hata fark edilmemişti. Bu sürümde atama açık
> biçimde yapılır.


In [ ]:
def png_series_to_dicom(png_dir, reference_dicom_dir, output_dir, size=(512, 512)):
    """PNG serisini, orijinal DICOM serisinin meta verilerini kullanarak DICOM'a çevirir.

    Eşleştirme dosya adı üzerinden yapılır: `<ad>.png` ↔ `<ad>` (uzantısız DICOM).
    """
    os.makedirs(output_dir, exist_ok=True)

    # Orijinal DICOM dosyalarının InstanceNumber'ları
    dicom_files = {}
    for root, _dirs, files in os.walk(reference_dicom_dir):
        for name in files:
            if not name.startswith("."):
                dicom_files[name] = os.path.join(root, name)

    original_numbers = {}
    for name, path in dicom_files.items():
        try:
            dataset = pydicom.dcmread(path, stop_before_pixels=True)
            original_numbers[name] = int(dataset.get("InstanceNumber", 0) or 0)
        except Exception:
            original_numbers[name] = 0

    def sort_key(png_name):
        return original_numbers.get(os.path.splitext(png_name)[0], 0)

    converted = 0
    sorted_pngs = sorted(
        (f for f in os.listdir(png_dir) if f.lower().endswith(".png")),
        key=sort_key,
    )

    for instance_number, png_name in enumerate(sorted_pngs, start=1):
        stem = os.path.splitext(png_name)[0]
        reference_path = dicom_files.get(stem)
        if reference_path is None:
            print(f"Uyarı: {png_name} için referans DICOM bulunamadı, atlandı.")
            continue

        original = pydicom.dcmread(reference_path)
        image = Image.open(os.path.join(png_dir, png_name)).convert("RGB").resize(size)

        new_dicom = copy.deepcopy(original)
        new_dicom.PixelData = np.array(image).tobytes()
        # Pillow .size -> (genişlik, yükseklik); DICOM -> Rows=yükseklik, Columns=genişlik
        new_dicom.Columns, new_dicom.Rows = image.size
        new_dicom.SamplesPerPixel = 3
        new_dicom.PhotometricInterpretation = "RGB"
        new_dicom.BitsAllocated = 8
        new_dicom.BitsStored = 8
        new_dicom.HighBit = 7
        new_dicom.PixelRepresentation = 0
        new_dicom.InstanceNumber = instance_number

        if not hasattr(new_dicom, "file_meta") or new_dicom.file_meta is None:
            new_dicom.file_meta = pydicom.dataset.FileMetaDataset()
        new_dicom.file_meta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian
        new_dicom.file_meta.MediaStorageSOPClassUID = new_dicom.SOPClassUID
        new_dicom.file_meta.MediaStorageSOPInstanceUID = new_dicom.SOPInstanceUID

        new_dicom.save_as(os.path.join(output_dir, stem + ".dcm"))
        converted += 1

    print(f"{converted} kesit DICOM olarak kaydedildi -> {output_dir}")
    return converted


converted_count = png_series_to_dicom(OUTPUT_DIR, DICOM_ARCHIVE, RESULT_DICOM_DIR)


## 7. Sonuç serisinin doğrulanması

Dışa aktarılan seri yeniden okunur ve tek bir görsel düzende gösterilir.

> **Önceki sürümdeki sorun:** doğrulama adımı bir `ipywidgets` kaydırıcısı
> kullanıyordu. Widget durumu notebook dosyasına yazılır ve GitHub üzerinde
> görüntülenemez. Bu sürüm statik bir ızgara kullanır.


In [ ]:
def preview_dicom_series(dicom_dir, count=6, images_per_row=3):
    """DICOM serisinden eşit aralıklı `count` kesiti ızgara olarak gösterir."""
    files = sorted(os.listdir(dicom_dir))
    if not files:
        print("Klasörde DICOM dosyası yok.")
        return

    step = max(1, len(files) // count)
    selected = files[::step][:count]

    rows = int(np.ceil(len(selected) / images_per_row))
    plt.figure(figsize=(5 * images_per_row, 5 * rows))

    for i, name in enumerate(selected, start=1):
        dataset = pydicom.dcmread(os.path.join(dicom_dir, name))
        pixels = dataset.pixel_array
        if pixels.ndim == 3 and pixels.shape[2] == 3:
            display = pixels
        else:
            display = pixels
        plt.subplot(rows, images_per_row, i)
        plt.imshow(display, cmap=None if display.ndim == 3 else "gray")
        plt.title(name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


preview_dicom_series(RESULT_DICOM_DIR)


## 8. Özet

Pipeline tamamlandığında elde edilen dosya sayıları:


In [ ]:
summary = {
    "DICOM -> PNG": len(os.listdir(PNG_DIR)),
    "normal": len(os.listdir(NORMAL_DIR)),
    "anormal": len(os.listdir(ABNORMAL_DIR)),
    "segmentlenen": len(os.listdir(SEGMENTED_DIR)),
    "birleşik çıktı": len(os.listdir(OUTPUT_DIR)),
    "sonuç DICOM": len(os.listdir(RESULT_DICOM_DIR)),
}

for label, value in summary.items():
    print(f"{label:<20}: {value:>5}")

assert summary["DICOM -> PNG"] == summary["birleşik çıktı"], (
    "Birleşik çıktı kesit sayısı girdiyle eşleşmiyor — eksik kesit var."
)
print()
print("Tutarlılık doğrulandı: çıktı serisi girdiyle aynı sayıda kesit içerir.")


## Sonuçlar ve çıkarımlar

* Pipeline, ham bir DICOM arşivinden PACS'te açılabilir bir sonuç serisi üretir.
* Akış **yıkıcı değildir**: kaynak DICOM arşivine hiçbir adımda yazılmaz veya
  silinmez; tüm çıktılar `cikarim/` altındadır.
* Sonuç serisi orijinal serinin meta verilerini (hasta bilgileri, çekim
  parametreleri) taşır; ancak `SOPInstanceUID` gibi tanımlayıcılar da kopyalanır.
  Üretim kullanımında yeni bir seri UID üretilmesi ve serinin "türetilmiş" olarak
  işaretlenmesi (`DerivationDescription`) gerekir.
* Arayüz sürümü (`app/lungscan_gui.py`) bu akışı aynı adımlarla uygular ve ek
  olarak ilerleme göstergesi, kesit önizlemesi ve iptal desteği sunar.

### Sınırlamalar

* Sınıflandırıcı kesit düzeyinde karar verir; kesitin hangi hastaya veya hangi
  anatomik bölgeye ait olduğunu bilmez. Aynı lezyon komşu kesitlerde tekrar tekrar
  işaretlenebilir.
* Sonuçlar otomatik bir ön taramadır; **radyolog onayı olmadan klinik kullanım
  için uygun değildir** (bkz. `DISCLAIMER.md`).
* Segmentasyon ve sınıflandırma modelleri farklı veri kümelerinde eğitilmiştir;
  olasılık kalibrasyonu yapılmamıştır. Bu nedenle çıktıdaki güven skorları
  olasılık olarak değil, **sıralama ölçütü** olarak yorumlanmalıdır.
